# Protein Qwen3.5-Style Pretraining From Scratch

This notebook is protein-only. It reads compiled `train.txt` lines in the form `<|protein|>...<|endoftext|>`, builds the repo protein tokenizer from scratch, loads a Qwen3.5 backbone config through `libs`, adapts that config to the MDC decoder, then trains from random initialization.

In [ ]:
import json
import math
import sys
from pathlib import Path

import torch

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "libs").is_dir():
            return path
    raise RuntimeError("Could not locate project root from the current notebook directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from libs.core import (
    MDCDecoderModel,
    build_mdc_config_from_qwen3_5_config,
    build_or_load_protein_tokenizer,
    build_qwen3_5_config,
    compute_mdc_causal_lm_loss,
    count_trainable_parameters,
    create_protein_lm_dataloader,
    evaluate_mdc_causal_lm_batch_loss,
    generate_protein_text,
    load_protein_corpus_text,
    save_protein_pretrain_checkpoint,
    split_protein_corpus_text,
)

PROJECT_ROOT

In [ ]:
TRAIN_TEXT_PATH = PROJECT_ROOT / "data" / "compiled" / "refseq_bacteria_protein" / "train.txt"
TOKENIZER_MAP_PATH = TRAIN_TEXT_PATH.with_name("tokenizer_map.json")
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "checkpoints" / "protein_from_scratch"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

QWEN_MODEL_SIZE = "0.8B"
QWEN_CONFIG_OVERRIDES = {
    "emb_dim": 256,
    "n_heads": 4,
    "n_layers": 4,
    "hidden_dim": 1024,
    "head_dim": 64,
    "n_kv_groups": 2,
    "linear_key_head_dim": 64,
    "linear_value_head_dim": 64,
    "linear_num_key_heads": 4,
    "linear_num_value_heads": 4,
}
CONTEXT_LENGTH = 512
STRIDE = 256
TOKENIZER_VOCAB_SIZE = 512
REBUILD_TOKENIZER = False

TRAIN_RATIO = 0.9
BATCH_SIZE = 2
NUM_EPOCHS = 1
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP_NORM = 1.0
EVAL_FREQ = 50
EVAL_BATCHES = 10

if not TRAIN_TEXT_PATH.exists():
    raise FileNotFoundError(
        f"Missing protein corpus: {TRAIN_TEXT_PATH}. Build it first with cmd/build_refseq_profile_text.py."
    )

config_summary = {
    "train_text_path": str(TRAIN_TEXT_PATH),
    "tokenizer_map_path": str(TOKENIZER_MAP_PATH),
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "qwen_model_size": QWEN_MODEL_SIZE,
    "qwen_config_overrides": QWEN_CONFIG_OVERRIDES,
    "context_length": CONTEXT_LENGTH,
    "tokenizer_vocab_size": TOKENIZER_VOCAB_SIZE,
}
config_summary

In [ ]:
corpus_text = load_protein_corpus_text(TRAIN_TEXT_PATH)
tokenizer_artifact = build_or_load_protein_tokenizer(
    TRAIN_TEXT_PATH,
    tokenizer_map_path=TOKENIZER_MAP_PATH,
    vocab_size=TOKENIZER_VOCAB_SIZE,
    rebuild=REBUILD_TOKENIZER,
)
train_text, val_text = split_protein_corpus_text(corpus_text, train_ratio=TRAIN_RATIO)

train_loader = create_protein_lm_dataloader(
    train_text,
    tokenizer_artifact.tokenizer,
    context_length=CONTEXT_LENGTH,
    stride=STRIDE,
    batch_size=BATCH_SIZE,
    shuffle=True,
)
val_loader = create_protein_lm_dataloader(
    val_text,
    tokenizer_artifact.tokenizer,
    context_length=CONTEXT_LENGTH,
    stride=STRIDE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print(f"Corpus chars: {len(corpus_text):,}")
print(f"Tokenizer vocab: {tokenizer_artifact.vocab_size} rebuilt={tokenizer_artifact.rebuilt}")
print(f"Batches: train={len(train_loader)} val={len(val_loader)}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
runtime_dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float32

qwen_config = build_qwen3_5_config(
    QWEN_MODEL_SIZE,
    vocab_size=tokenizer_artifact.vocab_size,
    context_length=CONTEXT_LENGTH,
    dtype=runtime_dtype,
    overrides=QWEN_CONFIG_OVERRIDES,
)

model_config = build_mdc_config_from_qwen3_5_config(
    qwen_config,
    dtype=runtime_dtype,
    attention_pattern="as_config",
)
model = MDCDecoderModel(model_config).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print(f"Device: {device} dtype={runtime_dtype}")
print(f"Trainable parameters: {count_trainable_parameters(model):,}")
model_config.to_dict()

In [ ]:
checkpoint_last_path = CHECKPOINT_DIR / "checkpoint_last.pt"
checkpoint_best_path = CHECKPOINT_DIR / "checkpoint_best.pt"
metrics_path = CHECKPOINT_DIR / "training_metrics.jsonl"

global_step = 0
tokens_seen = 0
train_losses = []
val_losses = []
best_val_loss = math.inf

def move_batch(batch, device):
    return type(batch)(
        input_ids=batch.input_ids.to(device),
        attention_mask=batch.attention_mask.to(device),
        labels=batch.labels.to(device),
    )

def append_metrics(payload):
    with metrics_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, ensure_ascii=False) + "\n")

def save_checkpoint(path, epoch, val_loss):
    return save_protein_pretrain_checkpoint(
        path,
        model=model,
        optimizer=optimizer,
        model_config=model_config,
        tokenizer=tokenizer_artifact.tokenizer,
        tokenizer_map_path=tokenizer_artifact.tokenizer_map_path,
        epoch=epoch,
        global_step=global_step,
        tokens_seen=tokens_seen,
        train_losses=train_losses,
        val_losses=val_losses,
        best_val_loss=None if math.isinf(best_val_loss) else best_val_loss,
        training_args={
            "batch_size": BATCH_SIZE,
            "context_length": CONTEXT_LENGTH,
            "stride": STRIDE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "qwen_model_size": QWEN_MODEL_SIZE,
            "qwen_config_overrides": QWEN_CONFIG_OVERRIDES,
        },
        extra={
            "corpus_file": str(TRAIN_TEXT_PATH.resolve()),
            "qwen3_5_config": dict(qwen_config),
            "last_eval_val_loss": val_loss,
        },
    )

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    for batch in train_loader:
        batch = move_batch(batch, device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch.input_ids, attn_mask=batch.attention_mask)
        loss = compute_mdc_causal_lm_loss(logits, batch.labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        global_step += 1
        tokens_seen += int(batch.attention_mask.sum().item())

        if global_step % EVAL_FREQ == 0:
            train_eval_loss = evaluate_mdc_causal_lm_batch_loss(
                model, train_loader, device=device, max_batches=EVAL_BATCHES
            )
            val_loss = evaluate_mdc_causal_lm_batch_loss(
                model, val_loader, device=device, max_batches=EVAL_BATCHES
            )
            train_losses.append(train_eval_loss)
            val_losses.append(val_loss)
            append_metrics({
                "epoch": epoch,
                "global_step": global_step,
                "tokens_seen": tokens_seen,
                "train_loss": train_eval_loss,
                "val_loss": val_loss,
            })
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                save_checkpoint(checkpoint_best_path, epoch, val_loss)
            print(f"epoch={epoch} step={global_step} train={train_eval_loss:.4f} val={val_loss:.4f}")

    val_loss = evaluate_mdc_causal_lm_batch_loss(model, val_loader, device=device, max_batches=EVAL_BATCHES)
    save_checkpoint(checkpoint_last_path, epoch, val_loss)

print(f"Saved last checkpoint: {checkpoint_last_path}")
print(f"Saved best checkpoint: {checkpoint_best_path if checkpoint_best_path.exists() else 'not yet'}")

In [ ]:
sample = generate_protein_text(
    model,
    tokenizer_artifact.tokenizer,
    prompt="<|protein|>",
    device=device,
    max_new_tokens=80,
    context_length=CONTEXT_LENGTH,
)
sample